In [1]:
import pandas as pd
import os
import numpy as np

# Загружаем первый источник (он уже сохранен локально)
df1 = pd.read_csv('data/rpl_raw.csv')

# Загружаем второй источник
path_understat = "/Users/aleksandr/.cache/kagglehub/datasets/codytipton/understat-data/versions/3"
df2 = pd.read_csv(os.path.join(path_understat, 'game_stats.csv'))

print("Оба файла успешно загружены в память!")

Оба файла успешно загружены в память!


In [2]:
# Проверяем структуру данных
print(df1.info())
print(df2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3416 entries, 0 to 3415
Data columns (total 19 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Country  3416 non-null   object 
 1   League   3416 non-null   object 
 2   Season   3416 non-null   object 
 3   Date     3416 non-null   object 
 4   Time     3416 non-null   object 
 5   Home     3416 non-null   object 
 6   Away     3416 non-null   object 
 7   HG       3416 non-null   int64  
 8   AG       3416 non-null   int64  
 9   Res      3416 non-null   object 
 10  PSCH     3229 non-null   float64
 11  PSCD     3229 non-null   float64
 12  PSCA     3229 non-null   float64
 13  MaxCH    3334 non-null   float64
 14  MaxCD    3334 non-null   float64
 15  MaxCA    3334 non-null   float64
 16  AvgCH    3334 non-null   float64
 17  AvgCD    3334 non-null   float64
 18  AvgCA    3334 non-null   float64
dtypes: float64(9), int64(2), object(8)
memory usage: 507.2+ KB
None
<class 'pandas.core.frame.

In [3]:
# Фильтруем данные второго источника по чемпионату РПЛ (Russian Premier League)
print(df2['league'].unique())  # Выводим уникальные значения в колонке 'league' для проверки
df2_rpl = df2[df2['league']=='RFPL'].copy()

print(df2_rpl.sample(5))  # Выводим случайные 5 строк для проверки

teams1 = df1['Home'].sort_values().unique()
print(teams1)
teams2 = df2_rpl['club_name'].sort_values().unique()
print(teams2)

set_teams1 = set(teams1)
set_teams2 = set(teams2)
incommon_teams1 = sorted(set_teams1-set_teams2)
incommon_teams2 = sorted(set_teams2-set_teams1)
print(f"Команды с разными названиями в первом списке:\n {incommon_teams1}")
print(f"Команды с разными названиями во втором списке:\n {incommon_teams2}")

['EPL' 'La liga' 'Bundesliga' 'Serie A' 'Ligue 1' 'RFPL']
        id league  season       club_name home_away        xG       xGA  \
38051  195   RFPL    2016           Amkar         h  1.125000  1.378090   
38608  200   RFPL    2017    Arsenal Tula         h  0.180377  0.970025   
38226  183   RFPL    2017  Spartak Moscow         a  2.664500  0.828105   
39772  187   RFPL    2020       FC Rostov         h  3.169480  1.682800   
37280  184   RFPL    2015          FC Ufa         a  1.418100  2.398310   

           npxG     npxGA       ppda  ...  scored  missed    xpts  result  \
38051  1.125000  1.378090  11.352941  ...       1       2  1.1207       l   
38608  0.180377  0.970025  16.476190  ...       0       0  0.4687       d   
38226  1.910220  0.828105   7.103448  ...       4       1  2.6635       w   
39772  3.169480  0.928520   9.764706  ...       3       0  2.3085       w   
37280  0.663830  1.644030  12.333333  ...       1       4  0.7075       l   

                      date  

In [4]:
# унифицируем названия команд
team_mapping = {
    # Из Understat -> В football-data
    'Dinamo Moscow': 'Dynamo Moscow',
    'FC Krasnodar': 'Krasnodar',
    'FC Rostov': 'FK Rostov',
    'FC Orenburg': 'Orenburg',
    'FC Ufa': 'Ufa',
    'FC Tambov': 'Tambov',
    'FC Rotor Volgograd': 'R. Volgograd',
    'FC Yenisey Krasnoyarsk': 'Yenisey',
    'FK Akhmat': 'Akhmat Grozny',
    'Fakel': 'Fakel Voronezh',
    'Krylya Sovetov Samara': 'Krylya Sovetov',
    'Kuban Krasnodar': 'Kuban',
    'Mordovya': 'M. Saransk',
    'Nizhny Novgorod': 'Pari NN',  # или как там было в первом списке
    'PFC Sochi': 'Sochi',
    'SKA-Khabarovsk': 'SKA Khabarovsk',
    'Tom Tomsk': 'Tomsk',
    'Zenit St. Petersburg': 'Zenit',
    'Anzhi Makhachkala': 'FK Anzi Makhackala',
    'Akron': 'Akron Togliatti',
    'Torpedo Moscow': 'T. Moscow'
}

df1['Home'] = df1['Home'].map(team_mapping).fillna(df1['Home'])
df1['Away'] = df1['Away'].map(team_mapping).fillna(df1['Away'])

df2_rpl['club_name'] = df2_rpl['club_name'].map(team_mapping).fillna(df2_rpl['club_name'])

print("Команды после унификации:")
print(df1['Home'].unique())
print(df1['Away'].unique())
print(df2_rpl['club_name'].unique())

Команды после унификации:
['M. Saransk' 'CSKA Moscow' 'Volga N. Novgorod' 'Vladikavkaz' 'Zenit'
 'Krylya Sovetov' 'FK Anzi Makhackala' 'Krasnodar' 'Kuban' 'Amkar'
 'Lokomotiv Moscow' 'Rubin Kazan' 'Spartak Moscow' 'FK Rostov'
 'Akhmat Grozny' 'Dynamo Moscow' 'SKA Khabarovsk' 'Spartak Nalchik' 'Ural'
 'Tomsk' 'Ufa' 'T. Moscow' 'Arsenal Tula' 'Tosno' 'Volgar-Astrakhan'
 'Orenburg' 'Yenisey' 'Tambov' 'Pari NN' 'Sochi' 'Khimki' 'R. Volgograd'
 'Fakel Voronezh' 'Rodina Moscow' 'Baltika' 'Akron Togliatti'
 'Dynamo Makhachkala']
['Lokomotiv Moscow' 'FK Rostov' 'Dynamo Moscow' 'Spartak Moscow' 'Amkar'
 'Akhmat Grozny' 'Kuban' 'Rubin Kazan' 'M. Saransk' 'CSKA Moscow'
 'Krylya Sovetov' 'Vladikavkaz' 'Volga N. Novgorod' 'FK Anzi Makhackala'
 'Krasnodar' 'Zenit' 'Spartak Nalchik' 'SKA Khabarovsk' 'Tomsk' 'Ural'
 'Ufa' 'T. Moscow' 'Arsenal Tula' 'Tosno' 'Volgar-Astrakhan' 'Orenburg'
 'Yenisey' 'Tambov' 'Pari NN' 'Sochi' 'Khimki' 'R. Volgograd'
 'Fakel Voronezh' 'Rodina Moscow' 'Baltika' 'Akron Togl

In [5]:
# Посмотрим на все столбцы df2_rpl
print("Все столбцы df2_rpl:")
print(df2_rpl.columns.tolist())

# Посмотрим на уникальные значения id
print(f"\nУникальных id в df2_rpl: {df2_rpl['id'].nunique()}")
print(f"Всего строк в df2_rpl: {df2_rpl.shape[0]}")

# Если id - это команда, то должно быть около 30-40 уникальных значений (количество команд РПЛ)
# Если id - это матч, то должно быть около 2000+ уникальных значений (количество матчей)

df1_home = pd.DataFrame({
    'date': df1['Date'],
    'season': df1['Season'],
    'club_name': df1['Home'],
    'home_away': 'h',
    'scored': df1['HG'],
    'result': df1['Res'].map({'H': 'W', 'D': 'D', 'A': 'L'})
})

df1_away = pd.DataFrame({
    'date': df1['Date'],
    'season': df1['Season'],
    'club_name': df1['Away'],
    'home_away': 'a',
    'scored': df1['AG'],
    'result': df1['Res'].map({'H': 'L', 'D': 'D', 'A': 'W'})
})

df1_long = pd.concat([df1_home, df1_away], ignore_index=True)

df1_long['date'] = pd.to_datetime(df1_long['date'], format='%d/%m/%Y')
df2_rpl['date'] = pd.to_datetime(df2_rpl['date']).dt.date  # Преобразуем в datetime.date для соответствия формату df1_long

print("--- Данные после преобразования в длинный формат ---")
print(df1_long.head())
print(df1_long.shape)
print(df2_rpl.head())
print(df2_rpl.shape)

Все столбцы df2_rpl:
['id', 'league', 'season', 'club_name', 'home_away', 'xG', 'xGA', 'npxG', 'npxGA', 'ppda', 'ppda_allowed', 'deep', 'deep_allowed', 'scored', 'missed', 'xpts', 'result', 'date', 'wins', 'draws', 'loses', 'pts', 'npxGD']

Уникальных id в df2_rpl: 32
Всего строк в df2_rpl: 4960
--- Данные после преобразования в длинный формат ---
        date     season          club_name home_away  scored result
0 2012-07-20  2012/2013         M. Saransk         h       2      L
1 2012-07-21  2012/2013        CSKA Moscow         h       1      W
2 2012-07-21  2012/2013  Volga N. Novgorod         h       1      W
3 2012-07-21  2012/2013        Vladikavkaz         h       1      L
4 2012-07-22  2012/2013              Zenit         h       2      W
(6832, 6)
        id league  season       club_name home_away        xG       xGA  \
36762  183   RFPL    2014  Spartak Moscow         a  2.191290  1.196590   
36763  183   RFPL    2014  Spartak Moscow         a  1.231400  0.470186   
36764  

In [6]:
# Отбираем колонки и объединяем данные
df1_long['date'] = pd.to_datetime(df1_long['date']).dt.date
df2_rpl['date'] = pd.to_datetime(df2_rpl['date']).dt.date

cols_to_keep = ['date', 'season', 'club_name', 'home_away', 'xG', 'xGA', 'result']
df2_clean=df2_rpl[cols_to_keep].copy()

df_merged = df1_long.merge(df2_clean, on=['date', 'club_name', 'home_away'], how='inner', suffixes=('_fd', '_us'))

print("--- Данные после объединения ---")
print(df1_long.shape)
print(df2_rpl.shape)
print(df_merged.head())
print(df_merged.shape)

--- Данные после объединения ---
(6832, 6)
(4960, 23)
         date  season_fd      club_name home_away  scored result_fd  \
0  2014-08-01  2014/2015    Rubin Kazan         h       0         L   
1  2014-08-02  2014/2015           Ural         h       2         L   
2  2014-08-02  2014/2015   Arsenal Tula         h       0         L   
3  2014-08-02  2014/2015    CSKA Moscow         h       4         W   
4  2014-08-03  2014/2015  Dynamo Moscow         h       7         W   

   season_us        xG       xGA result_us  
0       2014  1.196590  2.191290         l  
1       2014  1.156560  2.155570         l  
2       2014  0.767575  2.485540         l  
3       2014  3.138270  0.578158         w  
4       2014  4.814010  0.600496         w  
(4960, 10)


In [7]:
# Делаем полное внешнее объединение (outer join) с индикатором
df_check = pd.merge(
    df1_long[['date', 'season', 'club_name', 'home_away', 'scored']],
    df2_clean[['date', 'season', 'club_name', 'home_away', 'xG']],
    on=['date', 'club_name', 'home_away'],
    how='outer',
    indicator=True # Этот параметр добавит столбец '_merge'
)

# Оставляем только те строки, которые НЕ совпали в обоих источниках
# 'left_only' = есть в football-data, но нет в Understat
# 'right_only' = есть в Understat, но нет в football-data
missing_data = df_check[df_check['_merge'] != 'both']

print(f"Всего несовпавших строк: {missing_data.shape[0]}")
print("\n--- Список несовпавших матчей ---")
print(missing_data.sort_values(by=['date', 'club_name']).query("_merge == 'left_only'").head(5))
print(missing_data.sort_values(by=['date', 'club_name']).query("_merge == 'right_only'").head(5))

Всего несовпавших строк: 1872

--- Список несовпавших матчей ---
         date   season_x         club_name home_away  scored  season_y  xG  \
0  2012-07-20  2012/2013  Lokomotiv Moscow         a       3       NaN NaN   
1  2012-07-20  2012/2013        M. Saransk         h       2       NaN NaN   
2  2012-07-21  2012/2013       CSKA Moscow         h       1       NaN NaN   
3  2012-07-21  2012/2013     Dynamo Moscow         a       0       NaN NaN   
4  2012-07-21  2012/2013         FK Rostov         a       0       NaN NaN   

      _merge  
0  left_only  
1  left_only  
2  left_only  
3  left_only  
4  left_only  
Empty DataFrame
Columns: [date, season_x, club_name, home_away, scored, season_y, xG, _merge]
Index: []


In [8]:
# пытаемся понять, почему некоторые матчи не совпадают
print("Диапазон дат в football-data (df1_long):")
print(f"Мин: {df1_long['date'].min()}, Макс: {df1_long['date'].max()}")

print("\nДиапазон дат в Understat (df2_clean):")
print(f"Мин: {df2_clean['date'].min()}, Макс: {df2_clean['date'].max()}")

print("--- Матчи 'T. Moscow' в football-data (df1_long) ---")
torpedo_fd = df1_long[df1_long['club_name'] == 'T. Moscow'][['date', 'season', 'home_away', 'scored']].sort_values('date')
display(torpedo_fd.head(10))

print("\n--- Матчи 'T. Moscow' в Understat (df2_clean) ---")
torpedo_us = df2_clean[df2_clean['club_name'] == 'T. Moscow'][['date', 'season', 'home_away', 'xG']].sort_values('date')
display(torpedo_us.head(10))

Диапазон дат в football-data (df1_long):
Мин: 2012-07-20, Макс: 2026-08-02

Диапазон дат в Understat (df2_clean):
Мин: 2014-08-01, Макс: 2024-09-29
--- Матчи 'T. Moscow' в football-data (df1_long) ---


,date,season,home_away,scored
485,2014-05-18,2013/2014,h,2
3903,2014-05-22,2013/2014,a,0
3907,2014-08-02,2014/2015,a,1
3915,2014-08-09,2014/2015,a,1
504,2014-08-12,2014/2015,h,1
3928,2014-08-16,2014/2015,a,2
525,2014-08-24,2014/2015,h,0
3945,2014-08-30,2014/2015,a,0
3956,2014-09-14,2014/2015,a,1
551,2014-09-22,2014/2015,h,1



--- Матчи 'T. Moscow' в Understat (df2_clean) ---


,date,season,home_away,xG
37212,2014-08-02,2014,a,0.578158
37213,2014-08-09,2014,a,0.971363
37214,2014-08-12,2014,h,0.954046
37215,2014-08-16,2014,a,1.249040
37216,2014-08-24,2014,h,0.850627
37217,2014-08-30,2014,a,2.440060
37218,2014-09-14,2014,a,0.696589
37219,2014-09-22,2014,h,1.245530
37220,2014-09-29,2014,a,1.288600
37221,2014-10-20,2014,a,0.832131


In [9]:
# УНИФИКАЦИЯ ФОРМАТА СЕЗОНА
# В df2_rpl сезон записан как '2014'. Превращаем его в '2014/2015' как в df1_long.
df2_rpl['season'] = df2_rpl['season'].astype(int).astype(str) + '/' + (df2_rpl['season'].astype(int) + 1).astype(str)

print("Формат сезонов унифицирован.")
print("Примеры сезонов в df1_long:", df1_long['season'].unique()[:3])
print("Примеры сезонов в df2_rpl:", df2_rpl['season'].unique()[:3])

# СПРАВОЧНИК СЕЗОНОВ РПЛ
seasons_mapping = {
    '2014/2015': {'start': '2014-08-01', 'end': '2015-05-23'},
    '2015/2016': {'start': '2015-07-17', 'end': '2016-05-21'},
    '2016/2017': {'start': '2016-07-30', 'end': '2017-05-21'},
    '2017/2018': {'start': '2017-07-15', 'end': '2018-05-13'},
    '2018/2019': {'start': '2018-07-27', 'end': '2019-05-26'},
    '2019/2020': {'start': '2019-07-12', 'end': '2020-07-22'},  # COVID
    '2020/2021': {'start': '2020-08-08', 'end': '2021-05-16'},
    '2021/2022': {'start': '2021-07-23', 'end': '2022-05-21'},
    '2022/2023': {'start': '2022-07-15', 'end': '2023-06-03'},
    '2023/2024': {'start': '2023-07-21', 'end': '2024-05-25'}
}

# Превращаем словарь в DataFrame для удобного merge
seasons_df = pd.DataFrame([
    {'season': season, 'start_date': info['start'], 'end_date': info['end']}
    for season, info in seasons_mapping.items()
])
seasons_df['start_date'] = pd.to_datetime(seasons_df['start_date'])
seasons_df['end_date'] = pd.to_datetime(seasons_df['end_date'])

# ПРИМЕНЕНИЕ СПРАВОЧНИКА (MERGE)
df1_with_seasons = pd.merge(df1_long, seasons_df, on='season', how='left')
df2_with_seasons = pd.merge(df2_rpl, seasons_df, on='season', how='left')

# ФИЛЬТРАЦИЯ ПО ДАТАМ СЕЗОНА
# Оставляем только те матчи, дата которых попадает в официальный интервал регулярного чемпионата
df1_regular = df1_with_seasons[
    (df1_with_seasons['date'] >= df1_with_seasons['start_date']) & 
    (df1_with_seasons['date'] <= df1_with_seasons['end_date'])
].copy()

df2_regular = df2_with_seasons[
    (df2_with_seasons['date'] >= df2_with_seasons['start_date']) & 
    (df2_with_seasons['date'] <= df2_with_seasons['end_date'])
].copy()

print(f"\nСтрок в df1 после фильтрации по справочнику сезонов: {df1_regular.shape[0]}")
print(f"Строк в df2 после фильтрации по справочнику сезонов: {df2_regular.shape[0]}")

# ФИНАЛЬНАЯ ПРОВЕРКА СЛИЯНИЯ
df_check_final = pd.merge(
    df1_regular[['date', 'club_name', 'home_away']],
    df2_regular[['date', 'club_name', 'home_away']],
    on=['date', 'club_name', 'home_away'],
    how='outer',
    indicator=True
)

print(f"\nРезультат слияния после применения справочника сезонов:")
print(df_check_final['_merge'].value_counts())
print(df_check_final[df_check_final['_merge'] == 'left_only'].head(10))  # Выводим первые 10 строк для проверки
print(df_check_final[df_check_final['_merge'] == 'right_only'].tail(10))  # Выводим последние 10 строк для проверки

Формат сезонов унифицирован.
Примеры сезонов в df1_long: ['2012/2013' '2013/2014' '2014/2015']
Примеры сезонов в df2_rpl: ['2014/2015' '2015/2016' '2016/2017']

Строк в df1 после фильтрации по справочнику сезонов: 4774
Строк в df2 после фильтрации по справочнику сезонов: 4774

Результат слияния после применения справочника сезонов:
_merge
both          4774
left_only        0
right_only       0
Name: count, dtype: int64
Empty DataFrame
Columns: [date, club_name, home_away, _merge]
Index: []
Empty DataFrame
Columns: [date, club_name, home_away, _merge]
Index: []
